In [ ]:
from glob import glob
import cv2
import matplotlib.pyplot as plt
from subspaceadonnx import SubspaceAD

In [ ]:
# Prepare directories with good images and images with defects
NORMAL_IMG_DIR = "datasets/transistor/train/good"
TARGET_IMG_DIR = "datasets/transistor/test/bent_lead"

In [ ]:
model = SubspaceAD(
    "models/dinov3_vitsplus_224_pruned.onnx", # prepared model using export_onnx.py
    providers=["WebGpuExecutionProvider"]
)

### Fit model

In [ ]:
normal_imgs = [cv2.imread(f) for f in sorted(glob(f"{NORMAL_IMG_DIR}/*.png"))]
model.fit(normal_imgs)  # pass directory containing images or list[np.ndarray]
print(f"pixel threshold={model.threshold_:.3f}, image threshold={model.image_threshold_:.3f}")

In [ ]:
def visualize_anomaly_map(target_img, 
                          anomaly_map, 
                          image_score,
                          threshold=0.5):
    """
    target_img: BGR HWC
    anomaly_map: HW float32
    image_score: float
    threshold: float, anomaly_mapの閾値
    """
    fig, axs = plt.subplots(1, 4, figsize=(12, 4))
    axs[0].imshow(cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB))
    axs[0].set_title("Input Image")
    axs[1].imshow(anomaly_map, cmap="jet", vmin=0, vmax=1)
    axs[1].set_title(f"Anomaly Map (score={image_score:.3f})")
    axs[2].imshow(cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB))
    axs[2].imshow(anomaly_map, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    axs[2].set_title("Overlay")
    axs[3].imshow(anomaly_map >= threshold, cmap="gray")
    axs[3].set_title(f"Thresholded (>={threshold:.3f})")
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
target_imgs = [cv2.imread(f) for f in sorted(glob(f"{TARGET_IMG_DIR}/*.png"))]
for target_img in target_imgs[:5]:
    anomaly_map = model(target_img)
    image_score = float(anomaly_map.max())
    visualize_anomaly_map(
        target_img, anomaly_map, image_score, threshold=model.threshold_
    )

### Save & load fitted model

In [ ]:
# save PCA parameters to a file
pca_params = model.save_npz("models/pca_params.npz")
# load PCA parameters from a file
#model.load_npz("models/pca_params.npz")

### Run evaluation on MVTec Dataset

In [ ]:
from subspaceadonnx import MVTecEvaluator

evaluator = MVTecEvaluator(
    dataset_root="datasets",
    dataset_names=["screw", "leather"],
    onnx_path="models/dinov3_vitsplus_224_pruned.onnx",
    providers=["WebGpuExecutionProvider"]
)
result = evaluator.evaluate()
for key, v in result.items():
    print(f"{key}: {v}")